# Week 9 · Day 2 — Image Classification from a Folder, in PyTorch

**Yesterday** you built a digit classifier in **TensorFlow/Keras**. **Today** we do image classification in **PyTorch** — our main framework from here on — on a *real dataset that lives in folders on disk*.

This is how real image projects actually start: not with a tidy `load_data()`, but with **folders of image files**, one folder per class. Your job is to walk those folders, load the images, and turn them into something a network can learn from. That’s where Python’s **`os`** module does the heavy lifting.

### The dataset
A folder called `ImageClassification`, with one subfolder per class:
```
ImageClassification/
  Car/            car1.jpg, car2.jpg, ...
  Cricket ball/   ball1.jpg, ...
  Ice Cream Corn/ cone1.jpg, ...
```
Three classes, color photos. We’ll teach a neural network to tell them apart.

### Today’s plan
1. **Load images from folders with `os`** — the real-project skill.
2. Prepare the data (resize, flatten, scale, split).
3. **Learn PyTorch gently** — tensors, then *you* build the model and training loop.
4. Train and evaluate.
5. **Deep dive: activation functions.**
6. **Deep dive: optimizers.**

> Since PyTorch is still new to you, we go slow. Run every cell and read the comments.

---
## 1. Load the images from folders with `os`

The core idea, and the skill worth remembering:
- **`os.listdir(folder)`** lists what’s inside a folder — we use it once to find the **class names** (the subfolder names) and again to find the **image files** in each class.
- **`os.path.join(a, b)`** glues path pieces together correctly on any operating system (Windows uses `\`, Linux/Mac use `/` — `os.path.join` handles it so you don’t hardcode either).

We never type a single image filename by hand. The code discovers everything.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image   # Pillow: opens image files

# ---- the one path you may need to fix ----
DATA_DIR = "ImageClassification"

# os.listdir gives the subfolder names -> these ARE our class labels.
# we keep only actual folders, and sort so the order is always the same.
class_names = sorted([
    name for name in os.listdir(DATA_DIR)
    if os.path.isdir(os.path.join(DATA_DIR, name))
])

print("classes found:", class_names)
print("number of classes:", len(class_names))

> If `class_names` is empty or wrong, fix `DATA_DIR` above so it points at the folder that *contains* `Car`, `Cricket ball`, and `Ice Cream Corn`.

### Walk each class folder and load its images
We set a fixed image size (**64×64**, color) so every image becomes the same shape — a neural network needs uniform inputs. For each class folder we loop over its files, build the full path with `os.path.join`, open the image, resize it, and store it along with its label (the class index).

In [ ]:
IMG_SIZE = 64   # every image resized to 64x64
VALID_EXT = (".jpg", ".jpeg", ".png", ".bmp")   # accepted image types

X, y = [], []

for label_index, class_name in enumerate(class_names):
    class_folder = os.path.join(DATA_DIR, class_name)          # e.g. ImageClassification/Car
    files = os.listdir(class_folder)                           # all files in that class
    loaded = 0
    for filename in files:
        if not filename.lower().endswith(VALID_EXT):
            continue                                           # skip non-image files
        img_path = os.path.join(class_folder, filename)        # full path to this image
        try:
            img = Image.open(img_path).convert("RGB")          # open as color
            img = img.resize((IMG_SIZE, IMG_SIZE))             # make uniform size
            X.append(np.array(img))                            # (64, 64, 3) pixel array
            y.append(label_index)                              # this image's class
            loaded += 1
        except Exception as e:
            print(f"  skipped {filename}: {e}")                # corrupt file? skip it
    print(f"{class_name:16s}: loaded {loaded} images")

X = np.array(X, dtype="float32")
y = np.array(y)
print("\nX shape:", X.shape, "  (images, height, width, color channels)")
print("y shape:", y.shape)

### Look at the data first (always)
Before any modelling, see what we loaded — a few images from each class with their labels.

In [ ]:
plt.figure(figsize=(11, 4))
shown = 0
for label_index, class_name in enumerate(class_names):
    # find the first few images of this class to display
    idxs = np.where(y == label_index)[0][:4]
    for j, idx in enumerate(idxs):
        shown += 1
        plt.subplot(len(class_names), 4, label_index*4 + j + 1)
        plt.imshow(X[idx].astype("uint8"))   # uint8 for correct color display
        plt.title(class_name, fontsize=9)
        plt.axis("off")
plt.suptitle("A look at each class")
plt.tight_layout()
plt.show()

# class balance — how many images per class?
print("images per class:")
for label_index, class_name in enumerate(class_names):
    print(f"  {class_name:16s}: {(y == label_index).sum()}")

---
## 2. Prepare the data for the network

An ANN (a plain multi-layer network) takes a **flat vector**, not a 2D color image. So we:
1. **Flatten** each `64×64×3` image into one long vector of `64*64*3 = 12288` numbers.
2. **Scale** pixels from 0–255 down to 0–1 (networks train far better on small inputs). For image pixels, dividing by 255 is the standard, simple scaler.
3. **Split** into train and test.

> *Note:* flattening throws away the 2D shape of the image — which is why plain ANNs aren’t the best tool for images. Next week’s **CNNs** keep the 2D structure. But an ANN is a perfectly good place to learn, and it’ll still do well on three distinct classes.

In [ ]:
from sklearn.model_selection import train_test_split

# 1. flatten each image: (N, 64, 64, 3) -> (N, 12288)
X_flat = X.reshape(X.shape[0], -1)
print("flattened:", X_flat.shape, " (each image is now a vector of", X_flat.shape[1], "numbers)")

# 2. scale pixels 0-255 -> 0-1
X_flat = X_flat / 255.0

# 3. split into train / test (stratify keeps class balance in both sides)
X_train, X_test, y_train, y_test = train_test_split(
    X_flat, y, test_size=0.2, random_state=42, stratify=y)

print("train:", X_train.shape, "  test:", X_test.shape)
n_features = X_train.shape[1]   # 12288
n_classes = len(class_names)    # 3
print("inputs per image:", n_features, "  classes:", n_classes)

**The two numbers your model must match:**
- input size = **`n_features`** (12288 — the flattened pixels)
- output size = **`n_classes`** (3)

We use the variables `n_features` and `n_classes` so your model code stays correct even if you change the image size or add a class.

---
## 3. PyTorch, gently

### 3a. Tensors
A **tensor** is PyTorch’s NumPy array — same idea, but it can train (track gradients) and run on a GPU. Convert our prepared arrays into tensors: images as floats, labels as `long` integers.

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)   # integer class labels
X_test_t  = torch.tensor(X_test,  dtype=torch.float32)
y_test_t  = torch.tensor(y_test,  dtype=torch.long)

print("X_train_t:", X_train_t.shape, X_train_t.dtype)
print("y_train_t:", y_train_t.shape, y_train_t.dtype)
print("first label:", y_train_t[0].item(), "->", class_names[y_train_t[0].item()])

> **Labels stay integers.** In Keras yesterday you one-hot encoded the labels. PyTorch’s `CrossEntropyLoss` wants plain integer labels (`0, 1, 2`) and does the rest internally — simpler.

### 3b. Batches with `DataLoader`
We feed the network small **batches** rather than all images at once. `DataLoader` serves shuffled batches for us. (With a small dataset, a batch size of 32 is reasonable; lower it if you have very few images.)

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

train_ds = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)

print("training images:", len(train_ds))
print("batches per epoch:", len(train_loader))

### 3c. Build your network  ✍️ (Task A)

Now **you** write the model — you’ve seen the pattern all week. A PyTorch model is an `nn.Sequential` stack of `nn.Linear` layers with `nn.ReLU()` activations between them.

**Requirements:**
- First layer input must be **`n_features`**.
- At least **one hidden layer** with **`nn.ReLU()`** after it (more is fine).
- Last layer output must be **`n_classes`**. **No softmax** — `CrossEntropyLoss` handles that.

**Template to complete:**
```python
model = nn.Sequential(
    nn.Linear(n_features, 256),   # input -> first hidden
    nn.ReLU(),
    nn.Linear(256, 64),           # add/adjust hidden layers as you like
    nn.ReLU(),
    nn.Linear(64, n_classes)      # last layer -> n_classes, no activation
)
```

💡 *Shape rule: each layer’s output size must equal the next layer’s input size. Mismatches are the #1 beginner error.*

In [ ]:
torch.manual_seed(42)

# ===== YOUR CODE HERE (Task A) =====
# Build your network and name it exactly `model`.

model = None   # <-- replace with your nn.Sequential(...)

# ===================================

print(model)

### Self-check (provided ✅)
Confirms your model takes `n_features` in and gives `n_classes` out **before** you train. Fix Task A until this passes.

In [ ]:
assert model is not None, "Task A: you haven't built `model` yet."
with torch.no_grad():
    dummy = torch.randn(5, n_features)
    out = model(dummy)
assert out.shape == (5, n_classes), f"Output shape {tuple(out.shape)} should be (5, {n_classes}). Check first/last layer sizes."
print(f"✅ shape check passed: {n_features} in -> {n_classes} out. Ready to train.")

### 3d. Loss and optimizer
Two choices, like Keras’s `compile` — as separate objects:
- **loss:** `nn.CrossEntropyLoss()` (multi-class; integer labels + raw scores).
- **optimizer:** `Adam` (we’ll compare optimizers in Part 6).

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
print("loss and optimizer ready")

### 3e. Write the training loop  ✍️ (Task B)

In Keras `model.fit()` hid the loop. In PyTorch **you write it** — the same **four moves**, once per batch:
1. **forward** — `preds = model(xb)`
2. **loss** — `loss = loss_fn(preds, yb)`
3. **backward** — `optimizer.zero_grad()` then `loss.backward()`
4. **update** — `optimizer.step()`

We give you a reusable `train_model` function — **read it carefully**, it’s the whole point of today. (We provide this one so you can reuse it in the activation/optimizer experiments; make sure you understand every line.)

In [ ]:
def train_model(model, optimizer, loss_fn, loader,
                X_test_t, y_test_t, epochs=20, log=True):
    """Train a model; return per-epoch train loss and test accuracy."""
    train_losses, test_accs = [], []
    for epoch in range(epochs):
        model.train()
        running = 0.0
        for xb, yb in loader:
            preds = model(xb)              # 1. forward
            loss = loss_fn(preds, yb)      # 2. loss
            optimizer.zero_grad()          # 3. backward...
            loss.backward()
            optimizer.step()               # 4. update
            running += loss.item()
        train_losses.append(running / len(loader))

        model.eval()
        with torch.no_grad():
            acc = (model(X_test_t).argmax(1) == y_test_t).float().mean().item()
        test_accs.append(acc)
        if log:
            print(f"epoch {epoch+1:2d}  train loss {train_losses[-1]:.4f}  test acc {acc:.2%}")
    return train_losses, test_accs

---
## 4. Train and evaluate

In [ ]:
train_losses, test_accs = train_model(
    model, optimizer, loss_fn, train_loader, X_test_t, y_test_t, epochs=20)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(train_losses, color="purple", marker="o")
ax1.set_title("Training loss"); ax1.set_xlabel("epoch"); ax1.grid(alpha=0.3)
ax2.plot(test_accs, color="green", marker="o")
ax2.set_title("Test accuracy"); ax2.set_xlabel("epoch"); ax2.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# final test accuracy + confusion matrix
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

model.eval()
with torch.no_grad():
    test_pred = model(X_test_t).argmax(dim=1)
test_acc = (test_pred == y_test_t).float().mean().item()
print(f"TEST ACCURACY: {test_acc:.2%}")

cm = confusion_matrix(y_test_t, test_pred)
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(ax=ax, cmap="Blues", colorbar=False, xticks_rotation=30)
plt.title(f"Confusion matrix — {test_acc:.1%}")
plt.tight_layout()
plt.show()

In [ ]:
# look at some predictions — green = correct, red = wrong
n_show = min(10, len(X_test))
plt.figure(figsize=(13, 5))
for i in range(n_show):
    plt.subplot(2, 5, i+1)
    img = (X_test[i].reshape(IMG_SIZE, IMG_SIZE, 3) * 255).astype("uint8")
    plt.imshow(img)
    t, p = y_test_t[i].item(), test_pred[i].item()
    plt.title(f"{class_names[p]}\n(true: {class_names[t]})", color="green" if t == p else "red", fontsize=8)
    plt.axis("off")
plt.suptitle("Predictions")
plt.tight_layout()
plt.show()

**You just built a complete image classifier in PyTorch** — loaded raw image files from folders with `os`, prepared them, wrote the model and training loop, and evaluated on held-out images. That’s a real project.

Now the two ideas that decide how *well* a network like this trains.

---
## 5. Deep dive: Activation functions

An **activation function** is the small non-linear step inside each neuron (every `nn.ReLU()` in your model is one).

### Why we need them
Without an activation, stacking layers is pointless: a chain of purely linear steps collapses into **one** straight line, which can’t separate real-world classes. The activation adds the **bend** that lets a network model complex boundaries — the same reason the Week 8 XOR problem needed a hidden layer with activation.

The three you’ll meet most:

In [ ]:
z = torch.linspace(-6, 6, 200)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (name, out, desc) in zip(axes, [
    ("ReLU",    torch.relu(z),    "0 for negatives, linear for positives"),
    ("Sigmoid", torch.sigmoid(z), "squashes into (0, 1)"),
    ("Tanh",    torch.tanh(z),    "squashes into (-1, 1)")]):
    ax.plot(z, out, color="purple", linewidth=2)
    ax.axhline(0, color="gray", lw=0.5); ax.axvline(0, color="gray", lw=0.5)
    ax.set_title(name); ax.set_xlabel(desc, fontsize=9); ax.grid(alpha=0.3)
plt.suptitle("The three activation functions you'll use most")
plt.tight_layout()
plt.show()

| Activation | Output range | Good for | Watch out for |
|---|---|---|---|
| **ReLU** | 0 to ∞ | **hidden layers** (modern default) | “dying” neurons stuck at 0 |
| **Sigmoid** | 0 to 1 | a **binary** output (probability) | saturates — slows learning |
| **Tanh** | −1 to 1 | hidden layers (older nets, RNNs) | also saturates at extremes |

**Practical rule:** ReLU in hidden layers unless you have a reason not to. Sigmoid on the *output* for binary yes/no. For multi-class output (our case) use no final activation and let `CrossEntropyLoss` handle it.

### See it matter
Train the *same network* with three different hidden activations and compare. (Uses a helper so every version is identical except the activation.)

In [ ]:
def make_model(activation):
    """Same architecture, chosen hidden activation."""
    torch.manual_seed(42)
    return nn.Sequential(
        nn.Linear(n_features, 256), activation(),
        nn.Linear(256, 64),         activation(),
        nn.Linear(64, n_classes))

results_act = {}
for name, act in [("ReLU", nn.ReLU), ("Sigmoid", nn.Sigmoid), ("Tanh", nn.Tanh)]:
    m = make_model(act)
    opt = torch.optim.Adam(m.parameters(), lr=0.001)
    _, accs = train_model(m, opt, loss_fn, train_loader, X_test_t, y_test_t, epochs=15, log=False)
    results_act[name] = accs
    print(f"{name:8s} final test acc: {accs[-1]:.2%}")

In [ ]:
plt.figure(figsize=(9, 5))
for name, accs in results_act.items():
    plt.plot(range(1, len(accs)+1), accs, marker="o", label=name)
plt.xlabel("epoch"); plt.ylabel("test accuracy")
plt.title("Activation functions compared (same network, same data)")
plt.legend(); plt.grid(alpha=0.3)
plt.show()

Typically **ReLU and tanh climb faster** than **sigmoid**, whose gradients shrink when inputs are large (saturation), slowing early learning. That gap is why ReLU is the default in hidden layers. *(Exact numbers vary per run and depend on your images — the pattern is the lesson.)*

---
## 6. Deep dive: Optimizers

The **optimizer** is the rule that updates weights after `loss.backward()` computes gradients. Same job — reduce the loss — different ways of stepping:

- **SGD:** step straight downhill by gradient × learning rate. Simple, can be slow and zig-zaggy.
- **SGD + Momentum:** keep some speed from previous steps, like a ball rolling downhill. Smoother, faster.
- **Adam:** adapts the step size per weight automatically. Usually the fastest starter and the safest default.

Race them on the same network:

In [ ]:
def make_relu_model():
    torch.manual_seed(42)
    return nn.Sequential(
        nn.Linear(n_features, 256), nn.ReLU(),
        nn.Linear(256, 64),         nn.ReLU(),
        nn.Linear(64, n_classes))

optimizers = {
    "SGD":          lambda p: torch.optim.SGD(p, lr=0.01),
    "SGD+Momentum": lambda p: torch.optim.SGD(p, lr=0.01, momentum=0.9),
    "Adam":         lambda p: torch.optim.Adam(p, lr=0.001),
}

results_opt = {}
for name, make_opt in optimizers.items():
    m = make_relu_model()
    opt = make_opt(m.parameters())
    losses, _ = train_model(m, opt, loss_fn, train_loader, X_test_t, y_test_t, epochs=15, log=False)
    results_opt[name] = losses
    print(f"{name:14s} final train loss: {losses[-1]:.4f}")

In [ ]:
plt.figure(figsize=(9, 5))
for name, losses in results_opt.items():
    plt.plot(range(1, len(losses)+1), losses, marker="o", label=name)
plt.xlabel("epoch"); plt.ylabel("training loss")
plt.title("Optimizers compared (same network, same data)")
plt.legend(); plt.grid(alpha=0.3)
plt.show()

Usually **plain SGD** falls slowest, **Momentum** speeds it up, and **Adam** drops fastest early — which is why Adam is the common default. (On some problems a well-tuned SGD+Momentum generalizes better, so Adam isn’t always the final word. Knowing the trade-off is the skill.)

### The shared knob: learning rate
Every optimizer has a **learning rate** — the step size, and the single most important number to get roughly right. Too big overshoots; too small crawls.

In [ ]:
for lr in [0.0001, 0.001, 0.01, 0.1]:
    m = make_relu_model()
    opt = torch.optim.Adam(m.parameters(), lr=lr)
    losses, accs = train_model(m, opt, loss_fn, train_loader, X_test_t, y_test_t, epochs=8, log=False)
    print(f"lr={lr:<7}  final train loss {losses[-1]:.4f}   test acc {accs[-1]:.2%}")

Usually the middle learning rates do best; the smallest underfits in a few epochs and the largest can be unstable. Same lesson as Week 8 — now across optimizers.

---
## Your turn (practice) ✍️

Make small changes and observe. Pick at least two:

1. **Change the image size** to `IMG_SIZE = 32` (re-run from Part 1). Faster training — does accuracy drop much?
2. **Add a hidden layer** to your model in Task A. Does test accuracy improve, or start overfitting?
3. **Try `nn.LeakyReLU()`** in `make_model` and compare it to ReLU on the chart.
4. **Train longer** with SGD+Momentum (30–40 epochs) — does it catch Adam?

Write a sentence under each about what you saw.

In [ ]:
# ===== YOUR EXPERIMENTS HERE =====



---
## Summary

**Loading a real dataset from folders with `os`:**
- `os.listdir` discovers the class folders (the labels) and the image files inside each.
- `os.path.join` builds every path correctly — no hardcoded filenames, works on any OS.
- Images get **resized to a fixed size** and **flattened** so the ANN gets uniform vectors; pixels scaled by /255.

**PyTorch:**
- A **tensor** is a trainable NumPy array; labels stay **integer** with `CrossEntropyLoss` (no one-hot).
- `nn.Sequential` + `nn.Linear` + `nn.ReLU`, **no softmax** on the last layer.
- You **write the training loop**: forward → loss → backward → update, over `DataLoader` batches.

**Activation functions** — the non-linear bend inside neurons: **ReLU** (hidden default), **sigmoid** (binary output), **tanh** (older alternative). We saw ReLU learn faster than sigmoid.

**Optimizers** — the weight-update rule: **SGD** → **+Momentum** → **Adam** (adaptive, best default), all sharing the critical **learning rate** knob.

> A plain ANN flattens the image and loses its 2D structure. **Next week: CNNs**, built to keep that structure — and they’ll beat this comfortably on images.